# Marketing 3 — CRISP-DM workshop

**Business context:** Local grocery loyalty campaigns

**Problem:** Select eligible loyalty customers for a limited personalised offer campaign without assuming predicted conversion is causal campaign uplift.

**Proposed success criterion:** Compare with an all-nonconverter baseline; assess precision and recall at a 12% contact-capacity limit. Estimate illustrative contribution only if campaign margin and costs are justified; distinguish predicted conversion from incremental campaign impact.

**Dataset:** `marketing_3.csv`. These are synthetic teaching records, not real company data.

## 1–2. Confirm Business and Data Understanding
Explain the decision, stakeholder, prediction moment, observation unit, key and target. **Review or revise the supplied success criterion if it does not adequately represent the business decision.** State two evidenced findings and two limitations.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
DATA = Path('marketing_3.csv')
df = pd.read_csv(DATA)
print(df.shape)
display(df.head())
display(df.isna().sum().sort_values(ascending=False).head(12))
print('Exact duplicates:',df.duplicated().sum())

(2652, 12)


,customer_id,campaign_id,channel,discount,previous_orders_12m,days_since_last_order,avg_basket_eur_12m,loyalty_tier,campaign_cost_eur,clicks_after_campaign,purchase_revenue_after_campaign_eur,conversion_flag
0,M3C000000,CMP_M3,SMS,20.00,3,78,129.86,Silver,2.86,2,0.0,0
1,M3C000001,CMP_M3,push,0.10,11,375,30.98,Gold,1.16,3,0.0,0
2,M3C000002,CMP_M3,push,0.20,2,273,73.75,Silver,2.02,3,0.0,0
3,M3C000003,CMP_M3,Social,0.15,14,194,93.16,Silver,1.37,0,0.0,0
4,M3C000004,CMP_M3,push,0.05,7,109,29.66,Silver,1.41,3,0.0,0


avg_basket_eur_12m                     2
customer_id                            0
channel                                0
campaign_id                            0
discount                               0
previous_orders_12m                    0
days_since_last_order                  0
loyalty_tier                           0
campaign_cost_eur                      0
clicks_after_campaign                  0
purchase_revenue_after_campaign_eur    0
conversion_flag                        0
dtype: int64

Exact duplicates: 2


### Business and Data Understanding

**Decision:** The Campaign Manager must decide which eligible customer-campaign opportunities should receive a personalised offer, while contacting no more than 12% of eligible opportunities.

**Stakeholder:** Campaign Manager.

**Prediction moment:** Before the personalised campaign is sent.

**Observation unit:** One customer-campaign opportunity.

**Business key:** `customer_id` + `campaign_id`.

**Target:** `conversion_flag`, where 1 represents conversion and 0 represents no conversion.

**Revised success criterion:** The model should be compared with an all-nonconverter baseline, but accuracy alone is not sufficient. Because only 12% of eligible opportunities can be contacted, precision and recall within the selected top 12% are more relevant to the business decision.

**Finding 1:** The dataset contains 2,652 observations and 12 variables. It contains 2 exact duplicate rows, which could distort the analysis if counted as independent observations.

**Finding 2:** `avg_basket_eur_12m` contains 2 missing values. The preview also suggests inconsistent discount representation, with values such as 20.00 and 0.10 appearing in the same variable.

**Limitation 1:** `clicks_after_campaign` and `purchase_revenue_after_campaign_eur` are recorded after the campaign and therefore must not be used to predict conversion before the campaign is sent.

**Limitation 2:** The dataset shows whether conversion occurred, but it cannot prove that the campaign caused the conversion. Predicted conversion should therefore not be interpreted as incremental campaign uplift.

## 3. Preparation: three justified decisions
For EACH insight: evidence → technique → before/after verification → interpretation → next decision. Consider identifiers, leakage, duplicate keys, suspicious values, categories, date parsing and feature engineering. Explain why you retain, delete or combine variables. **Do not use the target or post-outcome fields as predictors.**

In [6]:
# Add your preparation code below. Inspect before/after evidence.
# Make a copy first:
work = df.copy()
print("Rows before:", len(work))
print("Exact duplicates before:", work.duplicated().sum())

work = work.drop_duplicates().copy()

print("Rows after:", len(work))
print("Exact duplicates after:", work.duplicated().sum())
print("\nChannel values:")
print(work["channel"].value_counts(dropna=False))

print("\nDiscount values:")
print(work["discount"].value_counts(dropna=False).sort_index())
# Standardise channel labels
work["channel"] = work["channel"].replace({"push": "Push"})

# Standardise discount to proportions
work.loc[work["discount"] > 1, "discount"] = (
    work.loc[work["discount"] > 1, "discount"] / 100
)

print("\nChannel values after:")
print(work["channel"].value_counts())

print("\nDiscount values after:")
print(work["discount"].value_counts().sort_index())
print("\nMissing values:")
print(work[["avg_basket_eur_12m", "campaign_cost_eur"]].isna().sum())

print("\nCampaign cost range:")
print(work["campaign_cost_eur"].describe())

print("\nNon-positive campaign costs:")
print(work.loc[work["campaign_cost_eur"] <= 0, "campaign_cost_eur"])
# Treat negative campaign costs as invalid/missing
work.loc[work["campaign_cost_eur"] < 0, "campaign_cost_eur"] = np.nan

print("\nAfter marking invalid costs as missing:")
print(work[["avg_basket_eur_12m", "campaign_cost_eur"]].isna().sum())

print("Negative campaign costs remaining:",
      (work["campaign_cost_eur"] < 0).sum())

Rows before: 2652
Exact duplicates before: 2
Rows after: 2650
Exact duplicates after: 0

Channel values:
channel
push      566
Social    550
Push      524
SMS       518
Email     492
Name: count, dtype: int64

Discount values:
discount
0.00     542
0.05     415
0.10     422
0.15     456
0.20     465
5.00      93
10.00     80
15.00     85
20.00     92
Name: count, dtype: int64

Channel values after:
channel
Push      1090
Social     550
SMS        518
Email      492
Name: count, dtype: int64

Discount values after:
discount
0.00    542
0.05    508
0.10    502
0.15    541
0.20    557
Name: count, dtype: int64

Missing values:
avg_basket_eur_12m    2
campaign_cost_eur     0
dtype: int64

Campaign cost range:
count    2650.000000
mean        2.574442
std         1.363413
min        -2.000000
25%         1.400000
50%         2.630000
75%         3.740000
max         4.900000
Name: campaign_cost_eur, dtype: float64

Non-positive campaign costs:
33    -2.0
79    -2.0
147   -2.0
Name: campaign

### Preparation decision log
| Phase 2 finding | Technique and reason | Verification | Risk | Next decision |
|---|---|---|---|---|
| 2 exact duplicate rows | Removed exact duplicates because repeated observations could give some records extra weight | Rows: 2652 → 2650; duplicates: 2 → 0 | Some repeated rows could theoretically represent real repeated events | Investigate inconsistent channel and discount values |
| Inconsistent `Push`/`push` labels and mixed discount scales | Standardised `push` to `Push` and converted percentage values above 1 to proportions by dividing by 100 | Channel categories were consolidated; discount now uses only 0.00, 0.05, 0.10, 0.15 and 0.20 | Assumes values such as 5, 10, 15 and 20 represent percentages rather than different measures | Investigate missing and implausible numeric values |
| 2 missing average-basket values and 3 negative campaign costs | Negative campaign costs were treated as missing because negative campaign costs are implausible; missing numeric values will be imputed using training data during modelling | Negative campaign costs: 3 → 0; 2 basket values and 3 campaign-cost values remain marked as missing | Assumes negative costs are data errors rather than legitimate credits or refunds | Impute numeric missing values within the training pipeline and exclude identifiers and post-campaign leakage fields from predictors |

## 4. Modelling
Define the prediction moment and allowed predictors. Use a defensible train/test split (chronological for forecasting). Fit learned preprocessing on TRAINING data only. Compare a simple baseline with a candidate model. Explain your feature engineering and excluded fields.

In [12]:
from sklearn.model_selection import train_test_split

features = [
    "channel",
    "discount",
    "previous_orders_12m",
    "days_since_last_order",
    "avg_basket_eur_12m",
    "loyalty_tier",
    "campaign_cost_eur"
]

target = "conversion_flag"

X = work[features]
y = work[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Predictors:", features)
print("Training observations:", len(X_train))
print("Test observations:", len(X_test))
print("Training conversion rate:", y_train.mean())
print("Test conversion rate:", y_test.mean())
from sklearn.metrics import accuracy_score, precision_score, recall_score

baseline_pred = np.zeros(len(y_test), dtype=int)

print("Baseline accuracy:", accuracy_score(y_test, baseline_pred))
print("Baseline precision:", precision_score(y_test, baseline_pred, zero_division=0))
print("Baseline recall:", recall_score(y_test, baseline_pred, zero_division=0))
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

numeric_features = [
    "discount",
    "previous_orders_12m",
    "days_since_last_order",
    "avg_basket_eur_12m",
    "campaign_cost_eur"
]

categorical_features = [
    "channel",
    "loyalty_tier"
]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

print("Model trained successfully.")
# Predict conversion probabilities
probabilities = model.predict_proba(X_test)[:, 1]

# Campaign Manager can contact only 12% of the test customers
n_contact = int(np.ceil(0.12 * len(X_test)))

# Rank customers from highest to lowest predicted probability
top_indices = np.argsort(probabilities)[::-1][:n_contact]

# Check how many selected customers actually converted
y_test_array = y_test.to_numpy()
selected_actual = y_test_array[top_indices]

top12_precision = selected_actual.mean()
top12_recall = selected_actual.sum() / y_test_array.sum()
overall_conversion_rate = y_test_array.mean()
lift = top12_precision / overall_conversion_rate

print("Test customers:", len(X_test))
print("Customers we can contact (12%):", n_contact)
print("Actual converters in test set:", y_test_array.sum())
print("Converters found in selected 12%:", selected_actual.sum())
print("Top 12% precision:", top12_precision)
print("Top 12% recall:", top12_recall)
print("Overall conversion rate:", overall_conversion_rate)
print("Lift over average:", lift)
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score

# Treat the selected top 12% as the customers predicted for contact
top12_pred = np.zeros(len(y_test), dtype=int)
top12_pred[top_indices] = 1

cm = confusion_matrix(y_test_array, top12_pred)

print("Confusion matrix:")
print(cm)

print("\nTop-12% decision accuracy:", accuracy_score(y_test_array, top12_pred))
print("Top-12% decision precision:", precision_score(y_test_array, top12_pred))
print("Top-12% decision recall:", recall_score(y_test_array, top12_pred))

tn, fp, fn, tp = cm.ravel()

print("\nTrue negatives:", tn)
print("False positives:", fp)
print("False negatives:", fn)
print("True positives:", tp)

Predictors: ['channel', 'discount', 'previous_orders_12m', 'days_since_last_order', 'avg_basket_eur_12m', 'loyalty_tier', 'campaign_cost_eur']
Training observations: 2120
Test observations: 530
Training conversion rate: 0.12641509433962264
Test conversion rate: 0.12641509433962264
Baseline accuracy: 0.8735849056603774
Baseline precision: 0.0
Baseline recall: 0.0
Model trained successfully.
Test customers: 530
Customers we can contact (12%): 64
Actual converters in test set: 67
Converters found in selected 12%: 14
Top 12% precision: 0.21875
Top 12% recall: 0.208955223880597
Overall conversion rate: 0.12641509433962264
Lift over average: 1.7304104477611941
Confusion matrix:
[[413  50]
 [ 53  14]]

Top-12% decision accuracy: 0.8056603773584906
Top-12% decision precision: 0.21875
Top-12% decision recall: 0.208955223880597

True negatives: 413
False positives: 50
False negatives: 53
True positives: 14


## 5. Evaluation
Report model vs baseline on held-out observations. Interpret relevant classification errors or forecasting MAE by business segment. Compare with your Phase 1 success criterion and discuss whether that criterion is realistic. If a required cost/impact is unavailable, state the assumption or request more data.

### Model Evaluation

The all-nonconverter baseline achieved 87.36% accuracy, but its precision and recall were both 0. This high accuracy is misleading because the majority of customers did not convert, and the baseline failed to identify any actual converters.

Under the 12% contact-capacity constraint, the model selected 64 of the 530 test observations. Of these, 14 actually converted, giving a precision of 21.88%. This is higher than the overall test conversion rate of 12.64%, producing a lift of approximately 1.73. The model therefore concentrated converters into the limited group available for contact better than selecting an average customer.

Recall at the 12% capacity limit was 20.90%, meaning the selected group captured 14 of the 67 actual converters. The confusion matrix showed 14 true positives, 50 false positives, 53 false negatives and 413 true negatives. False positives use limited campaign capacity without a conversion, while false negatives represent converters who were not selected.

The model's overall decision accuracy was 80.57%, lower than the all-nonconverter baseline. However, accuracy is not the most appropriate success measure for this business decision because the objective is to prioritise a small group of customers under a fixed 12% contact limit.

Overall, the model shows useful ranking performance, but the dataset does not establish incremental campaign uplift. Therefore, the results support using predicted conversion for prioritisation, not claiming that the campaign caused the conversions. A controlled experiment or additional causal data would be needed to estimate incremental campaign impact.

## 6. Deployment proposal (design only)
Specify who uses the prediction, when it runs, what action it supports, a human override, monitoring metric, feedback/refresh trigger and a condition for pausing use. A diagram or 6–8 sentences is sufficient. No production system required.

### Deployment Proposal

The Campaign Manager would use the model before each personalised offer campaign to prioritise eligible customers. The model would calculate a predicted conversion probability for each customer-campaign opportunity and rank customers from highest to lowest probability. Because campaign capacity is limited, only the top 12% would be recommended for contact.

The Campaign Manager should review the recommended list before launching the campaign and retain the ability to override individual selections when there is relevant business information not available to the model. Model performance should be monitored using precision, recall and lift at the 12% contact limit, rather than accuracy alone.

Actual campaign outcomes should be collected after each campaign and used to monitor whether performance changes over time. The model should be reviewed or retrained when performance declines or customer behaviour changes substantially. Deployment should be paused if data-quality problems occur or if precision and lift fall close to the normal conversion rate. Because the current data cannot establish causal campaign uplift, the model should support customer prioritisation only and should not be used to claim that the campaign caused additional conversions.

## Final group decision + individual accountability
**Group:** Pilot / Improve / Revisit earlier CRISP-DM phase / Do not deploy yet. Cite evidence.

**Individual:** Every member adds a named Markdown paragraph describing one preparation choice, one model finding and one business implication.

### Final Group Decision

**Decision: Pilot**

We recommend a limited pilot rather than full deployment. On the held-out test data, the model achieved 21.88% precision within the 12% contact-capacity limit, compared with an overall conversion rate of 12.64%. This represents a lift of approximately 1.73, showing that the model can concentrate more converters into the limited contact group.

However, the model captured only 20.90% of all actual converters, and the available data cannot demonstrate that the campaign itself caused the conversions. Therefore, the model should initially be used as a prioritisation tool in a controlled pilot. Performance should be monitored using precision, recall and lift at the 12% capacity limit before considering wider deployment.

### Individual Accountability

**Anastasia Kaplun**

My preparation contribution focused on identifying and correcting inconsistent data before modelling. I helped standardise the `channel` values by combining `push` and `Push`, and standardised the `discount` variable so that percentages such as 10 and proportions such as 0.10 were represented consistently. This was important because inconsistent values could cause the model to interpret equivalent campaign conditions as different.

The main model finding was that the selected top 12% of customers achieved a 21.88% conversion rate compared with the overall rate of 12.64%, producing a lift of approximately 1.73. From a business perspective, this suggests that the model can help the Campaign Manager prioritise limited campaign capacity more effectively. However, the result predicts conversion rather than causal campaign uplift, so I would support a controlled pilot rather than immediate full deployment.

